# Auswertung der Pipeline-Kennzahlen

Auswertung der Kennzahlen aus `data/stats.csv`.

Die Daten werden nach Status ausgewertet und als Tabellen
für die weitere Auswertung ausgegeben.

In [2]:
import csv
import statistics
from pathlib import Path

STATS_FILE = Path("data/stats.csv")

METRICS = [
    "enriched_objects",
    "cves",
    "software",
    "relationships"
]

LABELS = {
    "enriched_objects": "Objekte (angereichert)",
    "cves": "CVE-Objekte",
    "software": "Software-Objekte",
    "relationships": "Beziehungen"
}

## Daten einlesen

In [ ]:
def load_rows() -> list[dict]:
    #liest die Daten des letzten Verarbeitungslaufs
    with STATS_FILE.open(encoding="utf-8") as f:
        rows = list(csv.DictReader(f, delimiter=";"))

    if not rows:
        return []

    latest_run = max(row["run_at"] for row in rows)
    return [row for row in rows if row["run_at"] == latest_run]


rows = load_rows()

print(
    f"Datengrundlage: {STATS_FILE} - "
    f"{len(rows)} Meldungen "
)

Datengrundlage: data/stats.csv -250 Meldungen 


## Kennzahlen

In [ ]:
def to_int(value: str):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def describe(values: list[int]):
    return {
        "n": len(values),
        "min": min(values),
        "median": statistics.median(values),
        "mean": round(statistics.mean(values), 1),
        "max": max(values),
        "sum": sum(values),
    }


def print_table(title: str, rows: list[dict]):
    print(f"\n## {title}  (n = {len(rows)})\n")
    print("| Kennzahl | Min | Median | Mittelwert | Max | Summe |")
    print("|---|---|---|---|---|---|")

    for metric in METRICS:
        values = [to_int(r.get(metric)) for r in rows]
        values = [v for v in values if v is not None]

        if not values:
            print(f"| {LABELS[metric]} | – | – | – | – | – |")
            continue

        d = describe(values)

        print(
            f"| {LABELS[metric]} | "
            f"{d['min']} | {d['median']} | {d['mean']} | "
            f"{d['max']} | {d['sum']} |"
        )

## Auswertung nach Status

In [6]:
failed = [r for r in rows if not r.get("enriched_objects")]

if failed:
    print(
        f"Anreicherung nicht möglich (Basis-Fallback): "
        f"{len(failed)} Meldungen: "
        + ", ".join(r["wid_id"] for r in failed)
    )

ok = [r for r in rows if r.get("enriched_objects")]

by_status = {
    "NEU": [r for r in ok if r.get("status") == "NEU"],
    "UPDATE": [r for r in ok if r.get("status") == "UPDATE"],
    "ohne Statuskennzeichen": [
        r for r in ok
        if r.get("status") not in ("NEU", "UPDATE")
    ],
}

print_table("Gesamt", ok)

for status, subset in by_status.items():
    if subset:
        print_table(f"Status: {status}", subset)


## Gesamt  (n = 250)

| Kennzahl | Min | Median | Mittelwert | Max | Summe |
|---|---|---|---|---|---|
| Objekte (angereichert) | 3 | 39.0 | 452.5 | 11591 | 113125 |
| CVE-Objekte | 0 | 4.0 | 33.4 | 850 | 8344 |
| Software-Objekte | 1 | 7.0 | 9.5 | 74 | 2366 |
| Beziehungen | 0 | 24.0 | 407.7 | 11248 | 101915 |

## Status: NEU  (n = 46)

| Kennzahl | Min | Median | Mittelwert | Max | Summe |
|---|---|---|---|---|---|
| Objekte (angereichert) | 3 | 11.0 | 43.8 | 741 | 2017 |
| CVE-Objekte | 0 | 2.0 | 7.8 | 108 | 357 |
| Software-Objekte | 1 | 2.0 | 3.4 | 26 | 156 |
| Beziehungen | 0 | 4.0 | 30.7 | 684 | 1412 |

## Status: UPDATE  (n = 198)

| Kennzahl | Min | Median | Mittelwert | Max | Summe |
|---|---|---|---|---|---|
| Objekte (angereichert) | 7 | 52.0 | 560.8 | 11591 | 111046 |
| CVE-Objekte | 1 | 4.5 | 40.3 | 850 | 7972 |
| Software-Objekte | 1 | 9.0 | 11.1 | 74 | 2197 |
| Beziehungen | 2 | 31.0 | 507.5 | 11248 | 100481 |

## Status: ohne Statuskennzeichen  (n = 6)

| Kennzahl | M

## Basis vs. Anreicherung

In [7]:
base_total = sum(
    to_int(r.get("base_objects")) or 0
    for r in ok
)

enr_total = sum(
    to_int(r.get("enriched_objects")) or 0
    for r in ok
)

print("\n## Basis vs. Anreicherung\n")

print(
    f"- Objekte gesamt Basis: {base_total} | "
    f"angereichert: {enr_total} "
    f"(Faktor {round(enr_total / base_total, 1)})"
)

neu = by_status["NEU"]

if neu:
    med = statistics.median(
        to_int(r["enriched_objects"])
        for r in neu
    )

    print(
        f"- Median je NEU-Meldung: "
        f"Basis 2 Objekte -> angereichert {med} Objekte"
    )


## Basis vs. Anreicherung

- Objekte gesamt Basis: 500 | angereichert: 113125 (Faktor 226.2)
- Median je NEU-Meldung: Basis 2 Objekte -> angereichert 11.0 Objekte


## Größte Meldungen

In [8]:
top = sorted(
    ok,
    key=lambda r: to_int(r.get("enriched_objects")) or 0,
    reverse=True
)[:5]

print("\n## Größte Meldungen\n")
print("| WID-Kennung | Status | Objekte | CVEs | Software | Beziehungen |")
print("|---|---|---|---|---|---|")

for r in top:
    print(
        f"| {r['wid_id']} | "
        f"{r.get('status') or '–'} | "
        f"{r['enriched_objects']} | "
        f"{r['cves']} | "
        f"{r['software']} | "
        f"{r['relationships']} |"
    )


## Größte Meldungen

| WID-Kennung | Status | Objekte | CVEs | Software | Beziehungen |
|---|---|---|---|---|---|
| WID-SEC-2024-1188 | UPDATE | 11591 | 304 | 37 | 11248 |
| WID-SEC-2026-2852 | UPDATE | 7660 | 850 | 8 | 6800 |
| WID-SEC-2026-1700 | UPDATE | 6497 | 405 | 15 | 6075 |
| WID-SEC-2026-2403 | UPDATE | 6496 | 432 | 14 | 6048 |
| WID-SEC-2026-2077 | UPDATE | 5645 | 331 | 16 | 5296 |
